<a href="https://colab.research.google.com/github/alishbaallahditta99/Artifical-Intelligence-/blob/main/Task_4_Context_Aware_Chatbot_Using_LangChain_or_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faiss-cpu sentence-transformers transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 16.4 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

In [3]:
from google.colab import files
uploaded = files.upload()

pdf_name = list(uploaded.keys())[0]

reader = PdfReader(pdf_name)

text = ""
for page in reader.pages:
    text += page.extract_text()

Saving AI_notes.pdf to AI_notes.pdf


In [4]:
chunk_size = 500
chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)
embeddings = np.array(embeddings).astype("float32")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [7]:
llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

In [8]:
def get_chunks(query, k=3):
    q_emb = model.encode([query]).astype("float32")
    _, indices = index.search(q_emb, k)
    return [chunks[i] for i in indices[0]]

In [9]:
def ask(question):
    context = "\n".join(get_chunks(question))

    prompt = f"""
Answer using only context.

Context:
{context}

Question:
{question}

Answer:
"""

    return llm(prompt)[0]["generated_text"]

In [10]:
print(ask("What is this document about?"))

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer using only context.

Context:
e contents of our website and tutorials as timely and as precisely as 
possible, however, the contents may contain inaccuracies or errors. Tutorials Point  (I) Pvt. 
Ltd. provides no guarantee regarding the accuracy, timeliness or completeness of our website 
or its contents including this tutorial. If you discover any errors on our website or in this 
tutorial, please notify us at contact@tutorialspoint.com. 
  Artificial Intelligence  
  ii 
T able of Contents  
About the Tutorial ............
gence John McCarthy , it is “The science and 
engineering of making intelligent machines, especially intelligent computer programs”. 
Artificial Intelligence is a way of making a computer, a computer-controlled robot, or a 
software think intelligently, in the similar manner the intelligent humans think.  
AI is accomplished by studying how human brain thinks, and how humans learn, decide, and 
work while trying to solve a problem, and then using the outcom

In [11]:
test_questions = [
    "What is the main topic?",
    "Summarize the document",
    "Explain key points"
]

for q in test_questions:
    print("Q:", q)
    print("A:", ask(q))
    print("-"*50)

Q: What is the main topic?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: 
Answer using only context.

Context:
Artificial Intelligence  
  i 
About the T utorial 
This tutorial provides introductory knowledge on Artificial Intelligence. It would come to a 
great help if you are about to select Artificial Intelligence as a course subject. You can briefly 
know about the areas of AI in which research is prospering.  
Audience 
This tutorial is prepared for the students at beginner level who aspire to learn Artificial 
Intelligence.  
Prerequisites 
The basic knowledge of Computer Science is mandatory. The 
.. ................................ .......... 10 
Real Life Applications of Research Areas .................................................................................................... 11 
Task Classification of AI .............................................................................................................................. 12 Artificial Intelligence  
  iii 
4. AGENTS AND ENVIRONMENTS ................................ ............

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: 
Answer using only context.

Context:
ition 
Some intelligent systems are capable of hearing and comprehending the language in 
terms of sentences and their meanings while a human talks to it. It can handle different 
accents, slang words, noise in the background, change in human’s noise due to cold, 
etc.  
 
 Handwriting Recognition 
The handwriting recognition software reads the text written on paper by a pen or on 
screen by a stylus. It can recognize the shapes of the letters and convert it into editable 
text. 
 
 Intelligen
It is not well-organized or well-formatted. 
 It keeps changing constantly. 
AI Technique is a manner to organize and use the knowledge efficiently in such a way that: 
 It should be perceivable by the people who provide it. 
 It should be easily modifiable to correct errors. 
 It should be useful in many situations though it is incomplete or inaccurate. 
AI techniques elevate the speed of execution of the complex program it is equipped with.  
Appli

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: 
Answer using only context.

Context:
knowledge of Mathematics, 
Languages, Science, Mechanical or Electrical engineering is a plus.  
Disclaimer & Copyright 
 Copyright 2015 by Tutorials Point (I) Pvt. Ltd.  
All the content and graphics published in this e -book are the property of Tutorials Point (I) 
Pvt. Ltd. The user of this e -book is prohibited to reuse, retain, copy, distribute or republish 
any contents or a part of contents of this e -book in any manner without wri tten consent of 
the publisher.  
We strive to update th
lity of use and understand relationships in the 
absence of action or objects. Understanding complex 
and abstract ideas. 
Mathematicians, 
Scientists 
Spatial intelligence 
The ability to perceive visual or spatial information, 
change it, and re -create visual images without 
reference to the objects,  construct 3D images, and 
to move and rotate them. 
Map readers , 
Astronauts, 
Physicists 
Bodily-Kinesthetic 
intelligence 
The ability to use complet